In [ ]:
edges_to_remove = {
    tuple(sorted(["Xinjiang", "Heilongjiang"])),
    tuple(sorted(["Fujian", "Hainan"])),
    tuple(sorted(["Hainan", "Tibet"])),
    tuple(sorted(["Hainan", "Yunnan"])),
    tuple(sorted(["Heilongjiang", "Shanghai"])),
    tuple(sorted(["Jilin", "Shanghai"])),
    tuple(sorted(["Liaoning", "Shanghai"])),
    tuple(sorted(["Inner Mongolia", "Shanghai"])),
    tuple(sorted(["Junggar Basin", "Hailar Basin"])),
    tuple(sorted(["Yingen-Ejina Basin", "Hailar Basin"])),
    tuple(sorted(["Yingen-Ejina Basin", "Erlian Basin"])),
    tuple(sorted(["Erlian Basin", "Hailar Basin"])),
    tuple(sorted(["Junggar Basin", "Yingen-Ejina Basin"])),
    tuple(sorted(["Tibet", "Qiongdongnan Basin"])),
    tuple(sorted(["Yunnan", "Qiongdongnan Basin"])),
    tuple(sorted(["Pearl River Mouth Basin", "Qiongdongnan Basin"])),
    tuple(sorted(["Beibu Gulf Basin", "Qiongdongnan Basin"])),
    tuple(sorted(["Pearl River Mouth Basin", "East China Sea Basin"])),
    tuple(sorted(["Sanjiang Basin", "East China Sea Basin"])),
    tuple(sorted(["South Yellow Sea Basin", "East China Sea Basin"])),
    tuple(sorted(["Hailar Basin", "Sanjiang Basin"])),
    tuple(sorted(["South Yellow Sea Basin", "Sanjiang Basin"])),
    tuple(sorted(["North Yellow Sea Basin", "Sanjiang Basin"])),
    tuple(sorted(["Subei Basin", "South Yellow Sea Basin"])),
    tuple(sorted(["Bohai Bay Basin (offshore)", "South Yellow Sea Basin"])),
    tuple(sorted(["North Yellow Sea Basin", "South Yellow Sea Basin"])),
    tuple(sorted(["Bohai Bay Basin (offshore)", "North Yellow Sea Basin"])),
    tuple(sorted(["Bohai Bay Basin (onshore)", "Bohai Bay Basin (offshore)"])),
    tuple(sorted(["Subei Basin", "Bohai Bay Basin (offshore)"])),
}

In [ ]:
from scipy.spatial import Delaunay

points = province_and_basins[['longitude', 'latitude']].values
province_names = province_and_basins['name'].values
tri = Delaunay(points)

# Extract unique edges from triangle simplices
all_edges = set()
for simplex in tri.simplices:
    for i in range(3):
        all_edges.add(tuple(sorted([simplex[i], simplex[(i + 1) % 3]])))

# Edges to remove
edges_to_remove = {
    tuple(sorted(["Xinjiang", "Heilongjiang"])),
    tuple(sorted(["Fujian", "Hainan"])),
    tuple(sorted(["Hainan", "Tibet"])),
    tuple(sorted(["Hainan", "Yunnan"])),
    tuple(sorted(["Heilongjiang", "Shanghai"])),
    tuple(sorted(["Jilin", "Shanghai"])),
    tuple(sorted(["Liaoning", "Shanghai"])),
    tuple(sorted(["Inner Mongolia", "Shanghai"])),
    tuple(sorted(["Junggar Basin", "Hailar Basin"])),
    tuple(sorted(["Yingen-Ejina Basin", "Hailar Basin"])),
    tuple(sorted(["Yingen-Ejina Basin", "Erlian Basin"])),
    tuple(sorted(["Erlian Basin", "Hailar Basin"])),
    tuple(sorted(["Junggar Basin", "Yingen-Ejina Basin"])),
    tuple(sorted(["Tibet", "Qiongdongnan Basin"])),
    tuple(sorted(["Yunnan", "Qiongdongnan Basin"])),
    tuple(sorted(["Pearl River Mouth Basin", "Qiongdongnan Basin"])),
    tuple(sorted(["Beibu Gulf Basin", "Qiongdongnan Basin"])),
    tuple(sorted(["Pearl River Mouth Basin", "East China Sea Basin"])),
    tuple(sorted(["Sanjiang Basin", "East China Sea Basin"])),
    tuple(sorted(["South Yellow Sea Basin", "East China Sea Basin"])),
    tuple(sorted(["Hailar Basin", "Sanjiang Basin"])),
    tuple(sorted(["South Yellow Sea Basin", "Sanjiang Basin"])),
    tuple(sorted(["North Yellow Sea Basin", "Sanjiang Basin"])),
    tuple(sorted(["Subei Basin", "South Yellow Sea Basin"])),
    tuple(sorted(["Bohai Bay Basin (offshore)", "South Yellow Sea Basin"])),
    tuple(sorted(["North Yellow Sea Basin", "South Yellow Sea Basin"])),
    tuple(sorted(["Bohai Bay Basin (offshore)", "North Yellow Sea Basin"])),
    tuple(sorted(["Bohai Bay Basin (onshore)", "Bohai Bay Basin (offshore)"])),
    tuple(sorted(["Subei Basin", "Bohai Bay Basin (offshore)"])),
}

# Edges to add (beyond the Delaunay triangulation)
edges_to_add = {
    tuple(sorted(["Shandong", "South Yellow Sea Basin"])),
    tuple(sorted(["Shandong", "North Yellow Sea Basin"])),
}

name_to_idx = {name: idx for idx, name in enumerate(province_names)}

kept_edges = set()
removed_edges = set()
for i, j in all_edges:
    name_pair = tuple(sorted([province_names[i], province_names[j]]))
    if name_pair in edges_to_remove:
        removed_edges.add((i, j))
    else:
        kept_edges.add((i, j))

# Add extra edges by name
for a, b in edges_to_add:
    if a in name_to_idx and b in name_to_idx:
        edge = tuple(sorted([name_to_idx[a], name_to_idx[b]]))
        kept_edges.add(edge)
        removed_edges.discard(edge)

# Print kept pairs
kept_pairs = sorted([(province_names[i], province_names[j]) for i, j in kept_edges])
print(f"Kept {len(kept_pairs)} edges:")
for a, b in kept_pairs:
    print(f"  ({a}, {b})")

def edges_to_coords(edge_set):
    lons, lats = [], []
    for i, j in edge_set:
        lons.extend([points[i, 0], points[j, 0], None])
        lats.extend([points[i, 1], points[j, 1], None])
    return lons, lats

kept_lons, kept_lats = edges_to_coords(kept_edges)
removed_lons, removed_lats = edges_to_coords(removed_edges)

geojson_data = json.loads(gdf.to_json())

fig = go.Figure()

fig.add_trace(go.Choropleth(
    geojson=geojson_data,
    locations=gdf.index,
    z=[1] * len(gdf),
    colorscale=[[0, "#f4f4f4"], [1, "#f4f4f4"]],
    marker_line_color="black",
    marker_line_width=0.6,
    showscale=False,
    name="Provinces"
))

fig.add_trace(go.Scattergeo(
    lat=kept_lats,
    lon=kept_lons,
    mode="lines",
    line=dict(width=1, color="steelblue"),
    opacity=0.7,
    name="Kept Edges"
))

# fig.add_trace(go.Scattergeo(
#     lat=removed_lats,
#     lon=removed_lons,
#     mode="lines",
#     line=dict(width=2, color="red", dash="dash"),
#     opacity=0.8,
#     name="Removed Edges"
# ))

# Province capitals
fig.add_trace(go.Scattergeo(
    lat=df['latitude'],
    lon=df['longitude'],
    mode="markers",
    marker=dict(size=7, color="orange", line=dict(color="black", width=0.5)),
    text=df['name'],
    hovertemplate="Province: %{text}<br>Lat: %{lat}<br>Lon: %{lon}<extra></extra>",
    name="Province Capitals"
))

# Basin centroids
fig.add_trace(go.Scattergeo(
    lat=basins['latitude'],
    lon=basins['longitude'],
    mode="markers",
    marker=dict(size=7, color="mediumpurple", symbol="diamond", line=dict(color="black", width=0.5)),
    text=basins['name'],
    hovertemplate="Basin: %{text}<br>Lat: %{lat}<br>Lon: %{lon}<extra></extra>",
    name="CO2 Basins"
))

fig.update_geos(
    projection_type="mercator",
    fitbounds="locations",
    visible=False,
)

fig.update_layout(
    title="Delaunay Triangulation of Province Capitals + CO2 Basins",
    width=900,
    height=650,
    margin=dict(l=10, r=10, t=40, b=10)
)

fig.show()